# PDF Extraction Quality Check

## Goal

This notebook documents the first extraction-quality check for the Chekhov Sakhalin 1890 Census project.

The goal is to verify that the PDF text layer can be converted into a raw person-level sample before normalization and QA.

**MVP scope:** 500 raw person records extracted from the uploaded Korsakovskiy okrug PDF sample.

## Source and extraction method

Source file used for the MVP extraction:

```text
Korsakovskiy_okrug_test3(1).pdf
```

Extraction approach:

```text
searchable PDF text layer
↓
page-level text extraction
↓
preserve printed book page numbers
↓
remove obvious running headers / locality headings from person-level fields
↓
split text into person-level records
↓
create raw_extracted_sample_500.csv
```

The raw sample intentionally preserves source markup such as square brackets, angle brackets, punctuation, and archival note formats. Normalization is reviewed in the second notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
DATA_SAMPLE = PROJECT_ROOT / "data" / "sample"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
QA_DIR = PROJECT_ROOT / "outputs" / "qa"

raw_path = DATA_SAMPLE / "raw_extracted_sample_500.csv"
reparsed_path = DATA_SAMPLE / "raw_extracted_sample_500_reparsed.csv"
review_issues_path = DATA_SAMPLE / "raw_extraction_review_issues.csv"
extraction_summary_path = QA_DIR / "extraction_summary.md"

print("Project root:", PROJECT_ROOT)
print("Raw sample exists:", raw_path.exists())
print("Reparsed sample exists:", reparsed_path.exists())
print("Review issues file exists:", review_issues_path.exists())

Project root: /mnt/data/chekhov_notebook_repo
Raw sample exists: True
Reparsed sample exists: True
Review issues file exists: True


## Load raw extracted sample

The raw sample is the output of the extraction/splitting stage, before final normalization.

In [2]:
raw = pd.read_csv(raw_path)
raw.shape

(500, 19)

In [3]:
raw.head(3)

,settlement,page_number,household_number,legal_status,name_raw,family_status,age,religion,origin_place,arrival_year,occupation,literacy,marriage_status,allowance_status,illness,comments,notes_raw,source_record_number,source_block_raw
0,Маука,413,1.,Крестьянин из с[сыльных].,Иван Вардзинский,Хозяин,40.,Католич[еского].,Плоцкой.,1879.,Печник.,Грамотен.,NaN,Нет.,NaN,NaN,РГБ № 6810,1,2. 1. 3. Крестьянин из с[сыльных]. 4. Иван Вар...
1,Маука,413,2.,Поселенец.,Никита Панов,Хозяин,50.,Правосл[авного].,Самарск[ой].,NaN,NaN,Грамотен.,NaN,NaN,NaN,NaN,РГБ № 6811,2,2. 2. 3. Поселенец. 4. Никита Панов. Хозяин. 5...
2,Маука,413,3.,Поселенец.,Прохор Титов,Хозяин,34.,Правосл[авного].,Тульск[ой].,NaN,NaN,NaN,NaN,NaN,NaN,NaN,РГБ № 6812,3,2. 3. 3. Поселенец. 4. Прохор Титов. Хозяин. 5...


## Extraction summary

This section displays the extraction summary generated during the first sample run.

In [4]:
if extraction_summary_path.exists():
    print(extraction_summary_path.read_text(encoding="utf-8"))
else:
    print("No extraction_summary.md found. Use the cells below to reconstruct summary from the raw CSV.")

# Extraction Summary

Source PDF: `Korsakovskiy_okrug_test3(1).pdf`

PDF pages extracted: 35
Printed book pages covered: 413-447
Parsed person records: 573
Rows in sample CSV: 500
Rows flagged for manual review: 5

## Settlement counts

- Пост Корсаковский: 100
- Соловьевка: 69
- Поро-ан-Томари: 67
- Владимировка: 58
- Лютога: 53
- Большая Елань: 40
- Хомутовка: 38
- Маука: 38
- Третья Падь: 30
- Мицулька: 25
- Голый Мыс: 24
- Лиственичное: 15
- Вторая Падь: 10
- Первая Падь: 6

## Notes

This is a raw extraction sample. Values still preserve source markup such as square brackets, angle brackets, trailing punctuation, and unresolved source anomalies. Normalization and QA should be run in the next pipeline step.


## Basic structure checks

These checks confirm that the raw sample has the expected row count and source-derived fields.

In [5]:
expected_raw_columns = [
    "settlement", "page_number", "household_number", "legal_status", "name_raw",
    "family_status", "age", "religion", "origin_place", "arrival_year", "occupation",
    "literacy", "marriage_status", "allowance_status", "illness", "comments",
    "notes_raw", "source_record_number", "source_block_raw"
]

summary = pd.DataFrame({
    "check": [
        "row_count_is_500",
        "all_expected_columns_present",
        "page_number_present",
        "source_block_raw_present",
    ],
    "result": [
        len(raw) == 500,
        set(expected_raw_columns).issubset(raw.columns),
        raw["page_number"].notna().all(),
        raw["source_block_raw"].notna().all(),
    ]
})
summary

,check,result
0,row_count_is_500,True
1,all_expected_columns_present,True
2,page_number_present,True
3,source_block_raw_present,True


## Printed book page coverage

The project preserves printed book page numbers, not only PDF viewer page numbers. This is important for source traceability.

In [6]:
page_summary = raw.agg({"page_number": ["min", "max", "nunique"]}).T
page_summary

,min,max,nunique
page_number,413,444,31


In [7]:
raw.groupby("page_number").size().rename("records").reset_index().head(10)

,page_number,records
0,413,14
1,414,22
2,415,18
3,416,18
4,417,18
5,418,19
6,419,16
7,420,12
8,422,1
9,423,4


## Settlement coverage

This checks which localities are included in the 500-record MVP sample.

In [8]:
settlement_counts = (
    raw["settlement"]
    .value_counts()
    .rename_axis("settlement")
    .reset_index(name="records")
)
settlement_counts

,settlement,records
0,Пост Корсаковский,100
1,Соловьевка,69
2,Поро-ан-Томари,67
3,Лютога,53
4,Маука,38
5,Хомутовка,38
6,Третья Падь,30
7,Большая Елань,25
8,Мицулька,25
9,Голый Мыс,24


## Field completeness before normalization

This table helps identify which fields are often missing in the raw extraction. Missingness may reflect the original source, not necessarily parser failure.

In [9]:
missing_summary = (
    raw.isna().sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "field"})
)
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(raw) * 100).round(1)
missing_summary.sort_values("missing_pct", ascending=False)

,field,missing_count,missing_pct
14,illness,496,99.2
15,comments,445,89.0
10,occupation,400,80.0
9,arrival_year,179,35.8
12,marriage_status,152,30.4
11,literacy,119,23.8
13,allowance_status,40,8.0
8,origin_place,34,6.8
6,age,22,4.4
7,religion,14,2.8


## Manual review issues from raw extraction

The extraction stage may flag records whose category codes or field boundaries look suspicious. These are reviewed before final normalization.

In [10]:
if review_issues_path.exists():
    review_issues = pd.read_csv(review_issues_path)
    print("Review issue rows:", len(review_issues))
    display(review_issues.head(20))
else:
    print("No raw extraction review issue file found.")

Review issue rows: 5


,settlement,page_number,source_record_number,household_number,legal_status,name_raw,family_status,notes_raw,source_block_raw
0,Третья Падь,429,24,13.0,Дочь поселенца.,NaN,NaN,РГБ № 4568,2. 13. 3. Дочь поселенца. 5. Анна Андреева Руд...
1,Соловьевка,431,22,NaN,NaN,NaN,NaN,РГБ № 5421,2. 11. Сын сс[ыльнокаторжной]. 4. Иван Ланин. ...
2,Лютога,436,46,27.0,Аграфена Балаховская. Н[езаконнорожденная] дочь.,NaN,NaN,РГБ № 5775,2. 27. 3. Аграфена Балаховская. Н[езаконнорожд...
3,Лютога,436,47,NaN,NaN,Захар Балаховский,Н[езаконнорожденный] сын,РГБ № 5774,2. 27. 4. Захар Балаховский. Н[езаконнорожденн...
4,Лиственичное,440,4,4.0,Поселенец.,NaN,NaN,РГБ № 6473,2. 4. 3. Поселенец. 5. Егор Васильев Зипов. Хо...


## Raw source-block examples

These examples show why the raw extraction stage should be followed by normalization and QA. The source text may contain restored word parts, crossed-out text, footnote digits, and category-code typos.

In [11]:
for idx in [0, 127, 177, 244, 365, 422, 425]:
    if idx < len(raw):
        print("-" * 80)
        print(f"Row index: {idx}")
        print(raw.loc[idx, "source_block_raw"])

--------------------------------------------------------------------------------
Row index: 0
2. 1. 3. Крестьянин из с[сыльных]. 4. Иван Вардзинский. Хозяин. 5. 40. 6. Католич[еского]. 7. Плоцкой. 8. 1879. 9. Печник. 10. Грамотен. 12. Нет. РГБ № 6810.
--------------------------------------------------------------------------------
Row index: 127
2. 37. 3. Сс[ылно]каторжный. 4. Владимир Васильев Попов. Жилец. 5. 29. 6. Правосл[авного]. 7. Пензенск[ой]. 8. 1887. 9. Землемер. 10. Грамотен. 11. Холост. 12. Да. РГБ № 6982.
--------------------------------------------------------------------------------
Row index: 177
2. 30. 3. Поселенец. 4. Андрей Федоров Савельев. Хозяин. 5. 60. 6. 1882. 7. Симбирск[ой]. 8. 1882. 10. Грамотен. 11. Женат на родине. 12. Нет. РГБ № 5829.
--------------------------------------------------------------------------------
Row index: 244
2. 13. 3. Дочь поселенца. 5. Анна Андреева Рудзит. Дочь. 5. 4 м[есяца]. 6. Правосл[авного]. 7. На Сахалине. 12. Нет.13 РГБ № 4568

## Extraction-quality conclusion

The MVP extraction is usable for the next stage because:

- the PDF text layer produced page-level text;
- printed book page numbers were preserved;
- 500 person-level raw rows were created;
- raw source blocks were preserved for auditability;
- suspicious records were identifiable for manual review.

Proceed to:

```text
data/sample/raw_extracted_sample_500.csv
↓
normalization + manual anomaly review
↓
data/processed/clean_sample_500.csv
```